In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
import warnings


df = pd.read_csv('data/combined.csv')



In [ ]:

# class VolumeProfileAnalyzer:
#     """
#     Volume Profile分析工具类
    
#     用于计算、存储和可视化多日的Volume Profile数据
#     """
    
#     def __init__(self, data=None, time_column='t', price_columns=None, volume_column='v'):
#         """
#         初始化Volume Profile分析器
        
#         参数:
#         data: DataFrame, 包含OHLCV和时间戳数据
#         time_column: str, 时间戳列名
#         price_columns: dict, 价格列名映射，格式为{'open': 'o', 'high': 'h', 'low': 'l', 'close': 'c'}
#         volume_column: str, 成交量列名
#         """
#         self.time_column = time_column
#         self.volume_column = volume_column
        
#         # 设置默认价格列名
#         if price_columns is None:
#             self.price_columns = {'open': 'o', 'high': 'h', 'low': 'l', 'close': 'c'}
#         else:
#             self.price_columns = price_columns
            
#         # 存储结果的字典
#         self.results = {}
        
#         # 如果提供了数据，立即处理
#         if data is not None:
#             self.load_data(data)
    
#     def load_data(self, data):
#         """
#         加载并预处理数据
        
#         参数:
#         data: DataFrame, 包含OHLCV和时间戳数据
#         """
#         # 检查必要的列是否存在
#         required_columns = [self.time_column, self.volume_column] + list(self.price_columns.values())
#         missing_columns = [col for col in required_columns if col not in data.columns]
        
#         if missing_columns:
#             raise ValueError(f"数据缺少必要的列: {', '.join(missing_columns)}")
        
#         # 复制数据以避免修改原始数据
#         self.data = data.copy()
        
#         # 确保时间戳列是datetime类型
#         if not pd.api.types.is_datetime64_any_dtype(self.data[self.time_column]):
#             self.data[self.time_column] = pd.to_datetime(self.data[self.time_column])
        
#         # 添加日期列
#         self.data['date'] = self.data[self.time_column].dt.date
        
#         # 获取所有唯一日期
#         self.dates = sorted(self.data['date'].unique())
        
#         print(f"数据加载完成，包含{len(self.dates)}个交易日，从{self.dates[0]}到{self.dates[-1]}")
    
#     def calculate_all(self, num_bins=30, value_area_pct=0.7, force_recalculate=False):
#         """
#         计算所有日期的Volume Profile
        
#         参数:
#         num_bins: int, 价格区间数量
#         value_area_pct: float, Value Area包含的成交量百分比(0-1)
#         force_recalculate: bool, 是否强制重新计算已有结果
        
#         返回:
#         self，用于链式调用
#         """
#         if not hasattr(self, 'data'):
#             raise ValueError("请先使用load_data方法加载数据")
        
#         # 按日期分组计算
#         for date in self.dates:
#             # 如果已经计算过且不强制重新计算，则跳过
#             if date in self.results and not force_recalculate:
#                 continue
                
#             # 获取当日数据
#             day_data = self.data[self.data['date'] == date]
            
#             # 计算当日的Volume Profile
#             self._calculate_single_day(day_data, date, num_bins, value_area_pct)
        
#         return self
    
#     def calculate_for_dates(self, dates, num_bins=30, value_area_pct=0.7, force_recalculate=False):
#         """
#         计算指定日期的Volume Profile
        
#         参数:
#         dates: list或单个日期，要计算的日期
#         num_bins: int, 价格区间数量
#         value_area_pct: float, Value Area包含的成交量百分比(0-1)
#         force_recalculate: bool, 是否强制重新计算已有结果
        
#         返回:
#         self，用于链式调用
#         """
#         if not hasattr(self, 'data'):
#             raise ValueError("请先使用load_data方法加载数据")
        
#         # 确保dates是列表
#         if not isinstance(dates, list):
#             dates = [dates]
        
#         # 转换字符串日期为datetime.date对象
#         processed_dates = []
#         for d in dates:
#             if isinstance(d, str):
#                 try:
#                     d = datetime.strptime(d, '%Y-%m-%d').date()
#                 except ValueError:
#                     warnings.warn(f"无法解析日期: {d}，将被跳过")
#                     continue
#             processed_dates.append(d)
        
#         # 过滤出存在于数据中的日期
#         valid_dates = [d for d in processed_dates if d in self.dates]
#         if len(valid_dates) < len(processed_dates):
#             warnings.warn(f"有{len(processed_dates) - len(valid_dates)}个日期不在数据范围内")
        
#         # 计算每个有效日期的VP
#         for date in valid_dates:
#             # 如果已经计算过且不强制重新计算，则跳过
#             if date in self.results and not force_recalculate:
#                 continue
                
#             # 获取当日数据
#             day_data = self.data[self.data['date'] == date]
            
#             # 计算当日的Volume Profile
#             self._calculate_single_day(day_data, date, num_bins, value_area_pct)
        
#         return self
    
#     def _calculate_single_day(self, day_data, date, num_bins, value_area_pct):
#         """
#         计算单日的Volume Profile (内部方法)，增强了对NaN值的处理
#         """
#         # 获取价格列名
#         """计算单日的Volume Profile (内部方法)"""
#     # 检查数据条数
#         if len(day_data) < 2:  # 设置最小条数要求
#             warnings.warn(f"日期 {date} 的数据条数不足: {len(day_data)} 条")
#             self.results[date] = {
#                 'vp': None,
#                 'poc': None,
#                 'value_area': None,
#                 'ohlc': None,
#                 'error': f"数据条数不足({len(day_data)}条)"
#             }
#             return
            

#         h_col = self.price_columns['high']
#         l_col = self.price_columns['low']
#         c_col = self.price_columns['close']
#         o_col = self.price_columns['open']
#         v_col = self.volume_column
        
#         # 过滤掉包含NaN的行
#         day_data = day_data.dropna(subset=[h_col, l_col, c_col, v_col])
        
#         # 检查过滤后是否还有数据
#         if len(day_data) == 0:
#             warnings.warn(f"日期 {date} 的数据全部为NaN，无法计算Volume Profile")
#             self.results[date] = {
#                 'vp': None,
#                 'poc': None,
#                 'value_area': None,
#                 'ohlc': None,
#                 'error': "数据全部为NaN"
#             }
#             return
        
#         # 确定价格范围
#         min_price = day_data[l_col].min()
#         max_price = day_data[h_col].max()
        
#         # 检查价格范围是否有效
#         if pd.isna(min_price) or pd.isna(max_price) or min_price >= max_price:
#             warnings.warn(f"日期 {date} 的价格范围无效: min={min_price}, max={max_price}")
#             self.results[date] = {
#                 'vp': None,
#                 'poc': None,
#                 'value_area': None,
#                 'ohlc': {
#                     'open': day_data[o_col].iloc[0] if not day_data.empty else None,
#                     'high': max_price,
#                     'low': min_price,
#                     'close': day_data[c_col].iloc[-1] if not day_data.empty else None,
#                     'volume': day_data[v_col].sum()
#                 },
#                 'error': "价格范围无效"
#             }
#             return
        
#         # 创建价格区间
#         price_bins = np.linspace(min_price, max_price, num_bins + 1)
#         bin_centers = (price_bins[:-1] + price_bins[1:]) / 2
        
#         # 初始化成交量数组
#         volumes = np.zeros(num_bins)
        
#         # 分配成交量到价格区间
#         for _, row in day_data.iterrows():
#             # 使用收盘价确定区间
#             price = row[c_col]
#             volume = row[v_col]
            
#             # 跳过NaN值
#             if pd.isna(price) or pd.isna(volume):
#                 continue
            
#             # 找到对应的价格区间
#             if min_price <= price <= max_price:
#                 # 安全计算bin_idx，确保是整数
#                 bin_ratio = (price - min_price) / (max_price - min_price)
#                 bin_idx = min(int(bin_ratio * num_bins), num_bins - 1)
#                 volumes[bin_idx] += volume
        
#         # 创建结果DataFrame
#         volume_profile = pd.DataFrame({
#             'price': bin_centers,
#             'volume': volumes
#         })
        
#         # 检查是否有成交量
#         if np.sum(volumes) == 0:
#             warnings.warn(f"日期 {date} 没有有效成交量")
#             self.results[date] = {
#                 'vp': volume_profile,
#                 'poc': None,
#                 'value_area': None,
#                 'ohlc': {
#                     'open': day_data[o_col].iloc[0] if not day_data.empty else None,
#                     'high': max_price,
#                     'low': min_price,
#                     'close': day_data[c_col].iloc[-1] if not day_data.empty else None,
#                     'volume': 0
#                 },
#                 'error': "没有有效成交量"
#             }
#             return
        
#         # 计算POC
#         poc_idx = np.argmax(volumes)
#         poc = bin_centers[poc_idx]
        
#         # 计算Value Area
#         total_volume = np.sum(volumes)
#         target_volume = total_volume * value_area_pct
        
#         # 从POC开始向两边扩展
#         sorted_idx = np.argsort(-volumes)  # 按成交量降序排列
#         cumulative_volume = 0
#         included_idx = []
        
#         for idx in sorted_idx:
#             included_idx.append(idx)
#             cumulative_volume += volumes[idx]
#             if cumulative_volume >= target_volume:
#                 break
        
#         value_area = [min(bin_centers[included_idx]), max(bin_centers[included_idx])]
        
#         # 存储OHLC信息
#         ohlc = {
#             'open': day_data[o_col].iloc[0] if not day_data.empty else None,
#             'high': max_price,
#             'low': min_price,
#             'close': day_data[c_col].iloc[-1] if not day_data.empty else None,
#             'volume': total_volume
#         }
        
#         # 存储当日结果
#         self.results[date] = {
#             'vp': volume_profile,
#             'poc': poc,
#             'value_area': value_area,
#             'ohlc': ohlc,
#             'params': {
#                 'num_bins': num_bins,
#                 'value_area_pct': value_area_pct
#             }
#         }

    
#     def get_result(self, date):
#         """
#         获取指定日期的Volume Profile结果
        
#         参数:
#         date: datetime.date或str，日期
        
#         返回:
#         dict, 包含vp、poc、value_area和ohlc的字典，如果日期不存在则返回None
#         """
#         # 如果是字符串，转换为日期对象
#         if isinstance(date, str):
#             try:
#                 date = datetime.strptime(date, '%Y-%m-%d').date()
#             except ValueError:
#                 warnings.warn(f"无法解析日期: {date}")
#                 return None
        
#         # 返回结果，如果不存在则返回None
#         return self.results.get(date)
    
#     def plot(self, dates=None, num_days=3, figsize=(10, 6), show_ohlc=True):
#         """
#         绘制Volume Profile图表
        
#         参数:
#         dates: list或单个日期，要绘制的日期，如果为None则使用最近的num_days天
#         num_days: int, 当dates为None时，绘制最近的天数
#         figsize: tuple, 图表大小
#         show_ohlc: bool, 是否显示OHLC信息
        
#         返回:
#         fig, axes: 图表对象和轴对象
#         """
#         if not self.results:
#             raise ValueError("没有计算结果，请先调用calculate_all或calculate_for_dates方法")
        
#         # 确定要绘制的日期
#         if dates is None:
#             # 使用最近的num_days天
#             plot_dates = sorted(self.results.keys())[-num_days:]
#         else:
#             # 确保dates是列表
#             if not isinstance(dates, list):
#                 dates = [dates]
            
#             # 转换字符串日期
#             plot_dates = []
#             for d in dates:
#                 if isinstance(d, str):
#                     try:
#                         d = datetime.strptime(d, '%Y-%m-%d').date()
#                     except ValueError:
#                         warnings.warn(f"无法解析日期: {d}，将被跳过")
#                         continue
#                 plot_dates.append(d)
            
#             # 过滤出有结果的日期
#             plot_dates = [d for d in plot_dates if d in self.results]
        
#         if not plot_dates:
#             raise ValueError("没有有效的日期可以绘制")
        
#         # 创建子图
#         fig, axes = plt.subplots(len(plot_dates), 1, figsize=(figsize[0], figsize[1]*len(plot_dates)), sharex=False)
#         if len(plot_dates) == 1:
#             axes = [axes]  # 确保axes始终是列表
        
#         # 为每个日期绘制Volume Profile
#         for i, date in enumerate(plot_dates):
#             result = self.results[date]
#             vp = result['vp']
#             poc = result['poc']
#             value_area = result['value_area']
#             ohlc = result['ohlc']
            
#             # 绘制横向条形图
#             ax = axes[i]
#             bar_height = (vp['price'].max() - vp['price'].min()) / len(vp)
#             ax.barh(vp['price'], vp['volume'], height=bar_height, color='skyblue', alpha=0.7)
            
#             # 标记POC
#             poc_volume = vp.loc[vp['price'].sub(poc).abs().idxmin(), 'volume']
#             ax.barh(poc, poc_volume, height=bar_height, color='red', alpha=0.7)
            
#             # 标记Value Area
#             ax.axhline(y=value_area[0], color='green', linestyle='--', alpha=0.7)
#             ax.axhline(y=value_area[1], color='green', linestyle='--', alpha=0.7)
            
#             # 设置标题和标签
#             title = f"Volume Profile - {date}"
#             if show_ohlc:
#                 ohlc_text = f"O:{ohlc['open']:.2f} H:{ohlc['high']:.2f} L:{ohlc['low']:.2f} C:{ohlc['close']:.2f} V:{ohlc['volume']:.0f}"
#                 title += f" - {ohlc_text}"
#             ax.set_title(title)
#             ax.set_xlabel('Volume')
#             ax.set_ylabel('Price')
            
#             # 添加POC和Value Area标注
#             ax.text(vp['volume'].max()*0.8, poc, f' POC: {poc:.2f}', 
#                     verticalalignment='center', color='red')
#             ax.text(vp['volume'].max()*0.8, value_area[0], f' VA Low: {value_area[0]:.2f}', 
#                     verticalalignment='bottom', color='green')
#             ax.text(vp['volume'].max()*0.8, value_area[1], f' VA High: {value_area[1]:.2f}', 
#                     verticalalignment='top', color='green')
            
#             # 设置Y轴范围略微超出当日价格范围
#             price_range = ohlc['high'] - ohlc['low']
#             ax.set_ylim(ohlc['low'] - price_range*0.05, ohlc['high'] + price_range*0.05)
        
#         plt.tight_layout()
#         plt.show()
        
#         return





In [51]:
# import numpy as np
# import pandas as pd
# import matplotlib.pyplot as plt
# from scipy import stats
# import warnings
# from datetime import date, datetime

# class MarketDataAnalyzer:
#     """
#     整合成交量分析和分布检测功能的市场数据分析器
#     """
#     def __init__(self, data=None, price_columns=None, volume_column='volume', 
#                  confidence_level=0.95, min_samples=30):
#         # 基础配置
#         self.price_columns = price_columns or {'open': 'open', 'high': 'high', 'low': 'low', 'close': 'close'}
#         self.volume_column = volume_column
#         self.confidence_level = confidence_level
#         self.min_samples = min_samples
        
#         # 数据存储
#         self.data = pd.DataFrame() if data is None else data.copy()
#         self.dates = []
#         self.results = {}
        
#         # 分布分析相关
#         self.daily_data = {}
#         self.distribution_params = {}
#         self.is_normal = {}
        
#         # 初始化处理
#         if not self.data.empty:
#             self._init_data()
    
#     def _init_data(self):
#         """初始化数据处理"""
#         # 确保日期列存在
#         if 'date' not in self.data.columns:
#             if 'datetime' in self.data.columns:
#                 self.data['date'] = self.data['datetime'].dt.date
#             else:
#                 raise ValueError("数据中必须包含'date'或'datetime'列")
        
#         # 获取唯一日期
#         self.dates = sorted(self.data['date'].unique())
        
#         # 初始化每日数据字典
#         for date_val in self.dates:
#             day_data = self.data[self.data['date'] == date_val]
#             prices = day_data[self.price_columns['close']].values
#             self.daily_data[date_val] = prices.tolist()
    
#     #------------------------
#     # 数据质量检查功能
#     #------------------------
    
#     def filter_insufficient_data_days(self, min_records=2):
#         """过滤掉数据条数不足的日期"""
#         # 计算每个日期的数据条数
#         date_counts = self.data.groupby('date').size()
        
#         # 找出数据条数足够的日期
#         valid_dates = date_counts[date_counts >= min_records].index.tolist()
        
#         # 记录被过滤的日期
#         filtered_dates = date_counts[date_counts < min_records].index.tolist()
#         if filtered_dates:
#             print(f"已过滤 {len(filtered_dates)} 个数据不足的日期:")
#             for date_val in filtered_dates:
#                 print(f"- {date_val}: {date_counts[date_val]} 条记录")
        
#         # 更新日期列表和数据
#         self.dates = [date_val for date_val in self.dates if date_val in valid_dates]
#         self.data = self.data[self.data['date'].isin(valid_dates)]
        
#         # 更新每日数据字典
#         for date_val in filtered_dates:
#             if date_val in self.daily_data:
#                 del self.daily_data[date_val]
        
#         return self
    
#     def check_data_quality(self, min_records=2, remove_invalid=True):
#         """检查数据质量并可选择性地移除无效日期"""
#         quality_issues = []
        
#         # 检查每个日期的数据
#         for date_val in self.dates.copy():  # 使用copy避免在迭代时修改
#             day_data = self.data[self.data['date'] == date_val]
            
#             issues = []
#             # 检查数据条数
#             if len(day_data) < min_records:
#                 issues.append(f"数据条数不足({len(day_data)}条)")
            
#             # 检查价格有效性
#             h_col = self.price_columns['high']
#             l_col = self.price_columns['low']
            
#             day_data_clean = day_data.dropna(subset=[h_col, l_col])
#             if day_data_clean.empty:
#                 issues.append("所有价格数据为NaN")
#             else:
#                 min_price = day_data_clean[l_col].min()
#                 max_price = day_data_clean[h_col].max()
                
#                 if pd.isna(min_price) or pd.isna(max_price):
#                     issues.append(f"价格包含NaN: min={min_price}, max={max_price}")
#                 elif min_price >= max_price:
#                     issues.append(f"价格范围无效: min={min_price}, max={max_price}")
            
#             # 如果有问题且需要移除
#             if issues and remove_invalid:
#                 self.dates.remove(date_val)
#                 if date_val in self.daily_data:
#                     del self.daily_data[date_val]
#                 quality_issues.append((date_val, issues))
#             elif issues:
#                 quality_issues.append((date_val, issues))
        
#         # 打印质量问题
#         if quality_issues:
#             print(f"发现 {len(quality_issues)} 个日期存在数据质量问题:")
#             for date_val, issues in quality_issues:
#                 status = "已移除" if remove_invalid else "保留"
#                 print(f"- {date_val} ({status}): {', '.join(issues)}")
        
#         return quality_issues
    
#     #------------------------
#     # 成交量分析功能
#     #------------------------
    
#     def calculate_volume_profile(self, num_bins=100, value_area_pct=0.68):
#         """计算所有日期的成交量分布"""
#         for date_val in self.dates:
#             day_data = self.data[self.data['date'] == date_val]
#             self._calculate_single_day_vp(day_data, date_val, num_bins, value_area_pct)
        
#         return self
    
#     def _calculate_single_day_vp(self, day_data, date_val, num_bins, value_area_pct):
#         """计算单日的Volume Profile (内部方法)"""
#         # 检查数据条数
#         if len(day_data) < 2:  # 设置最小条数要求
#             warnings.warn(f"日期 {date_val} 的数据条数不足: {len(day_data)} 条")
#             self.results[date_val] = {
#                 'vp': None,
#                 'poc': None,
#                 'value_area': None,
#                 'ohlc': None,
#                 'error': f"数据条数不足({len(day_data)}条)"
#             }
#             return
        
#         # 提取价格和成交量数据
#         h_col = self.price_columns['high']
#         l_col = self.price_columns['low']
#         o_col = self.price_columns['open']
#         c_col = self.price_columns['close']
#         v_col = self.volume_column
        
#         # 确保数据有效
#         day_data = day_data.dropna(subset=[h_col, l_col, v_col])
        
#         # 计算价格范围
#         min_price = day_data[l_col].min()
#         max_price = day_data[h_col].max()
        
#         if min_price >= max_price:
#             warnings.warn(f"日期 {date_val} 的价格范围无效: min={min_price}, max={max_price}")
#             self.results[date_val] = {
#                 'vp': None,
#                 'poc': None,
#                 'value_area': None,
#                 'ohlc': None,
#                 'error': f"价格范围无效: min={min_price}, max={max_price}"
#             }
#             return
        
#         # 创建价格区间
#         price_bins = np.linspace(min_price, max_price, num_bins + 1)
#         bin_width = (max_price - min_price) / num_bins
        
#         # 初始化成交量分布
#         volume_profile = np.zeros(num_bins)
        
#         # 计算每个交易在各价格区间的成交量分布
#         for _, row in day_data.iterrows():
#             h = row[h_col]
#             l = row[l_col]
#             v = row[v_col]
            
#             # 确定价格覆盖的区间
#             l_bin = max(0, int((l - min_price) / bin_width))
#             h_bin = min(num_bins - 1, int((h - min_price) / bin_width))
            
#             # 计算每个区间的成交量
#             if l_bin == h_bin:
#                 volume_profile[l_bin] += v
#             else:
#                 # 简单地将成交量平均分配到覆盖的价格区间
#                 covered_bins = h_bin - l_bin + 1
#                 vol_per_bin = v / covered_bins
#                 for bin_idx in range(l_bin, h_bin + 1):
#                     volume_profile[bin_idx] += vol_per_bin
        
#         # 找出POC (Point of Control)
#         poc_idx = np.argmax(volume_profile)
#         poc_price = min_price + (poc_idx + 0.5) * bin_width
        
#         # 计算Value Area
#         total_volume = np.sum(volume_profile)
#         target_volume = total_volume * value_area_pct
        
#         # 从POC开始向两侧扩展，直到覆盖目标成交量
#         left_idx, right_idx = poc_idx, poc_idx
#         current_volume = volume_profile[poc_idx]
        
#         while current_volume < target_volume and (left_idx > 0 or right_idx < num_bins - 1):
#             # 确定下一步是向左还是向右扩展
#             left_vol = volume_profile[left_idx - 1] if left_idx > 0 else 0
#             right_vol = volume_profile[right_idx + 1] if right_idx < num_bins - 1 else 0
            
#             if left_vol >= right_vol and left_idx > 0:
#                 left_idx -= 1
#                 current_volume += left_vol
#             elif right_idx < num_bins - 1:
#                 right_idx += 1
#                 current_volume += right_vol
#             else:
#                 break
        
#         # 计算Value Area的价格范围
#         va_low = min_price + left_idx * bin_width
#         va_high = min_price + (right_idx + 1) * bin_width
        
#         # 记录OHLC数据
#         ohlc = {
#             'open': day_data[o_col].iloc[0],
#             'high': max_price,
#             'low': min_price,
#             'close': day_data[c_col].iloc[-1]
#         }
        
#         # 存储结果
#         self.results[date_val] = {
#             'vp': {
#                 'bins': [(min_price + i * bin_width, min_price + (i + 1) * bin_width) for i in range(num_bins)],
#                 'volumes': volume_profile.tolist()
#             },
#             'poc': poc_price,
#             'value_area': (va_low, va_high),
#             'ohlc': ohlc
#         }
    
#     def plot_volume_profile(self, date_val, figsize=(12, 6)):
#         """绘制指定日期的成交量分布"""
#         if date_val not in self.results:
#             print(f"日期 {date_val} 的成交量分布尚未计算")
#             return
        
#         result = self.results[date_val]
#         if 'error' in result:
#             print(f"日期 {date_val} 的数据存在问题: {result['error']}")
#             return
        
#         vp = result['vp']
#         poc = result['poc']
#         va_low, va_high = result['value_area']
#         ohlc = result['ohlc']
        
#         # 创建图表
#         fig, ax = plt.subplots(figsize=figsize)
        
#         # 绘制成交量分布
#         bin_centers = [(b[0] + b[1]) / 2 for b in vp['bins']]
#         ax.barh(bin_centers, vp['volumes'], height=vp['bins'][0][1] - vp['bins'][0][0], 
#                 color='skyblue', alpha=0.7)
        
#         # 标记POC
#         ax.axhline(y=poc, color='red', linestyle='-', linewidth=1, label=f'POC: {poc:.2f}')
        
#         # 标记Value Area
#         ax.axhspan(va_low, va_high, alpha=0.2, color='green', label=f'Value Area: {va_low:.2f} - {va_high:.2f}')
        
#         # 标记OHLC
#         ax.axhline(y=ohlc['open'], color='blue', linestyle='--', linewidth=1, label=f'Open: {ohlc["open"]:.2f}')
#         ax.axhline(y=ohlc['close'], color='black', linestyle='--', linewidth=1, label=f'Close: {ohlc["close"]:.2f}')
        
#         # 设置标题和标签
#         ax.set_title(f'Volume Profile - {date_val}')
#         ax.set_xlabel('Volume')
#         ax.set_ylabel('Price')
#         ax.legend()
        
#         plt.tight_layout()
#         plt.show()
    
#     #------------------------
#     # 分布分析功能
#     #------------------------
    
#     def add_data_point(self, date_val, price, volume=None):
#         """添加新数据点并分析"""
#         # 处理日期格式
#         if isinstance(date_val, str):
#             date_val = datetime.strptime(date_val, '%Y-%m-%d').date()
        
#         # 初始化日期数据
#         if date_val not in self.daily_data:
#             self.daily_data[date_val] = []
#             if date_val not in self.dates:
#                 self.dates.append(date_val)
#                 self.dates.sort()
        
#         # 添加数据
#         self.daily_data[date_val].append(price)
        
#         # 创建新数据行并添加到DataFrame
#         new_row = {
#             'date': date_val,
#             self.price_columns['close']: price
#         }
#         if volume is not None:
#             new_row[self.volume_column] = volume
#         self.data = pd.concat([self.data, pd.DataFrame([new_row])], ignore_index=True)
        
#         # 检查数据点是否符合分布
#         result = self.check_point(date_val, price)
        
#         # 仅在首次达到最小样本量时计算分布参数
#         if len(self.daily_data[date_val]) == self.min_samples:
#             self.update_distribution(date_val)
        
#         return result

#     def update_distribution(self, date_val, force=False):
#         """更新指定日期的分布参数，可选是否强制更新"""
#         data = np.array(self.daily_data[date_val])
        
#         # 如果已经计算过且不强制更新，则跳过
#         if date_val in self.distribution_params and not force:
#             return self.is_normal.get(date_val, None)
        
#         # 检验是否符合正态分布（仅在数据足够时）
#         if len(data) >= 8:  # Shapiro-Wilk要求至少3个样本，但建议更多
#             shapiro_test = stats.shapiro(data)
#             self.is_normal[date_val] = shapiro_test.pvalue > 0.05
#         else:
#             self.is_normal[date_val] = None  # 数据不足，无法判断
        
#         # 计算分布参数
#         mean = np.mean(data)
#         std = np.std(data)
        
#         self.distribution_params[date_val] = {
#             'mean': mean,
#             'std': std,
#             'n': len(data),
#             'last_updated': len(data)  # 记录上次更新时的数据量
#         }
        
#         return self.is_normal[date_val]

#     def check_point(self, date_val, value):
#         """检查单个数据点是否符合当天分布"""
#         if date_val not in self.distribution_params:
#             return None  # 没有足够数据建立分布
        
#         params = self.distribution_params[date_val]
        
#         # 计算Z分数
#         z_score = (value - params['mean']) / params['std'] if params['std'] > 0 else 0
        
#         # 计算置信区间
#         alpha = 1 - self.confidence_level
#         critical_value = stats.norm.ppf(1 - alpha/2)  # 双尾检验
        
#         # 判断是否在置信区间内
#         is_within_interval = abs(z_score) <= critical_value
        
#         result = {
#             'value': value,
#             'z_score': z_score,
#             'critical_value': critical_value,
#             'is_within_interval': is_within_interval,
#             'is_normal_distribution': self.is_normal.get(date_val, None)
#         }
        
#         return result

#     def schedule_distribution_update(self, date_val, update_frequency=100):
#         """定期更新分布参数（例如每100个数据点）"""
#         if date_val not in self.distribution_params:
#             return False
        
#         current_count = len(self.daily_data[date_val])
#         last_updated = self.distribution_params[date_val]['last_updated']
        
#         if current_count - last_updated >= update_frequency:
#             self.update_distribution(date_val, force=True)
#             return True
        
#         return False



In [3]:
from multitimefram import MultiTimeframeVP

vp = MultiTimeframeVP()
vp.load_from_dataframe(df)


/Users/saika/Documents/git/gold/multitimefram.py:62: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  h4_key = timestamp.floor('4H')


In [5]:
from polygon import WebSocketClient
from polygon.websocket.models import WebSocketMessage, Market
from typing import List

client = WebSocketClient(market=Market.Forex,api_key='c7huTUjP2EF89VpEK5xKGAYSAktcSI8s')

# Aggregates (per minute)
# client.subscribe("CA.*") # all forex pair
client.subscribe("CA.USD/CAD")
client.subscribe("CA.USD/EUR")
client.subscribe("CA.USD/AUD")

# Aggregates (per second)
# client.subscribe("CAS.*") # all forex pair
# client.subscribe("CAS.USD/CAD")
# client.subscribe("CAS.USD/EUR")
# client.subscribe("CAS.USD/AUD")

# Quotes
# client.subscribe("C.*") # all forex pair
# client.subscribe("C.USD/CAD")
# client.subscribe("C.USD/EUR")
# client.subscribe("C.USD/AUD")


def handle_msg(msgs: List[WebSocketMessage]):
    for m in msgs:
        print(m)


client.run(handle_msg)

RuntimeError: asyncio.run() cannot be called from a running event loop